In [1]:
import pandas as pd
import implicit
from sklearn.model_selection import train_test_split
import scipy.sparse as sparse
from pandas.api.types import CategoricalDtype 
import numpy as np
import warnings
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore", category=UserWarning)

## Задание 1. Создание user-item-матрицы и разбиение данных на тест и контроль результатов

In [2]:
# Загрузка данных
ratings = pd.read_csv('ratings.csv')
ratings

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3
...,...,...,...
5976474,49925,510,5
5976475,49925,528,4
5976476,49925,722,4
5976477,49925,949,5


In [3]:
# Преобразование рейтингов в бинарные оценки
# Будем считать, если оценка больше 4, то книга понравилась
ratings['rating'] = ratings['rating'].apply(lambda x: 1 if x > 4 else 0)

In [4]:
# Пронумеруйте для каждого пользователя его прочитанные книги согласно расположению книг по порядку
ratings['book_order'] = ratings.groupby('user_id')['book_id'].rank(method='first')

In [5]:
# Переведите номера прочитанных книг в доли по формуле «порядковый номер / общее количество прочитанных пользователем книг»
ratings['book_order_ratio'] = ratings['book_order'] / ratings.groupby('user_id')['book_id'].transform('count')

In [6]:
ratings

,user_id,book_id,rating,book_order,book_order_ratio
0,1,258,1,63.0,0.538462
1,2,4081,0,54.0,0.830769
2,2,260,1,30.0,0.461538
3,2,9296,1,63.0,0.969231
4,2,2318,0,48.0,0.738462
...,...,...,...,...,...
5976474,49925,510,1,41.0,0.303704
5976475,49925,528,0,42.0,0.311111
5976476,49925,722,0,45.0,0.333333
5976477,49925,949,1,51.0,0.377778


In [7]:
# Разделите данные на обучение и контроль
train, test = train_test_split(ratings, train_size=0.8, random_state=42, shuffle=False)

In [8]:
# Построение user-item-матрицы
train_user_index = train.user_id.unique()
train_book_index = train.book_id.unique()
test_user_index = test.user_id.unique()
test_book_index = test.book_id.unique()

rows = train['user_id'].astype(pd.CategoricalDtype(categories=train_user_index)).cat.codes
cols = train['book_id'].astype(pd.CategoricalDtype(categories=train_book_index)).cat.codes
train_matrix = sparse.csr_matrix((train.rating, (rows, cols)), shape=(len(train_user_index), len(train_book_index)))

rows = test['user_id'].astype(pd.CategoricalDtype(categories=test_user_index)).cat.codes
cols = test['book_id'].astype(pd.CategoricalDtype(categories=test_book_index)).cat.codes
test_matrix = sparse.csr_matrix((test.rating, (rows, cols)), shape=(len(test_user_index), len(test_book_index)))


In [9]:
train_matrix = train_matrix.toarray()
test_matrix = test_matrix.toarray()

In [10]:
# Проверка размеров матриц
if train_matrix.shape[0] == len(train_user_index):
    print(f"Количество строк в обучающей матрице ({train_matrix.shape[0]}) РАВНО Количеству уникальных пользователей ({len(train_user_index)})")
else:
    print(f"Ошибка! Количество строк в обучающей матрице ({train_matrix.shape[0]}) НЕ РАВНО Количеству уникальных пользователей ({len(train_user_index)})")

if train_matrix.shape[1] == len(train_book_index):
    print(f"Количество столбцов в обучающей матрице ({train_matrix.shape[1]}) РАВНО Количеству уникальных книг ({len(train_book_index)})")
else:
    print(f"Ошибка! Количество столбцов в обучающей матрице ({train_matrix.shape[1]}) НЕ РАВНО Количеству уникальных книг ({len(train_book_index)})")

if test_matrix.shape[0] == len(test_user_index):
    print(f"Количество строк в тестовой матрице ({test_matrix.shape[0]}) РАВНО Количеству уникальных пользователей ({len(test_user_index)})")
else:
    print(f"Ошибка! Количество строк в тестовой матрице ({test_matrix.shape[0]}) НЕ РАВНО Количеству уникальных пользователей ({len(test_user_index)})")

if test_matrix.shape[1] == len(test_book_index):
    print(f"Количество столбцов в тестовой матрице ({test_matrix.shape[1]}) РАВНО Количеству уникальных книг ({len(test_book_index)})")
else:
    print(f"Ошибка! Количество столбцов в тестовой матрице ({test_matrix.shape[1]}) НЕ РАВНО Количеству уникальных книг ({len(test_book_index)})")


Количество строк в обучающей матрице (48769) РАВНО Количеству уникальных пользователей (48769)
Количество столбцов в обучающей матрице (9693) РАВНО Количеству уникальных книг (9693)
Количество строк в тестовой матрице (32281) РАВНО Количеству уникальных пользователей (32281)
Количество столбцов в тестовой матрице (9999) РАВНО Количеству уникальных книг (9999)


In [39]:
# Вывод списка названий книг с высокой оценкой для любого из пользователей
# Для удобства возьмем 2-го пользователя из списка
user_index = 2

In [12]:
user_id = train_user_index[user_index]
print(f"Cписок названий книг с высокой оценкой для пользователя {user_id}:")
for ind, score in enumerate(train_matrix[user_index]):
    if score>0:
        book_id = train_book_index[ind] 
        print(f'book_id={book_id}, score={score}')


Cписок названий книг с высокой оценкой для пользователя 4:
book_id=18, score=1
book_id=27, score=1
book_id=21, score=1
book_id=2, score=1
book_id=23, score=1
book_id=24, score=1
book_id=103, score=1
book_id=35, score=1
book_id=325, score=1
book_id=1237, score=1
book_id=36, score=1
book_id=102, score=1
book_id=693, score=1
book_id=87, score=1
book_id=42, score=1
book_id=245, score=1
book_id=81, score=1
book_id=10, score=1
book_id=497, score=1
book_id=489, score=1
book_id=1380, score=1
book_id=3469, score=1
book_id=85, score=1
book_id=321, score=1
book_id=93, score=1
book_id=383, score=1
book_id=1210, score=1
book_id=879, score=1
book_id=265, score=1
book_id=25, score=1
book_id=8464, score=1


## Задание 2. Создание бейзлайнов и расчёт метрик

In [13]:
def ap_at_k(recommendations, true_positives, k=10):
    """
    Подсчитывает метрику AP@K.

    Args:
        recommendations: Список рекомендаций (book_id).
        true_positives: Список книг, положительно оценённых пользователем.
        k: Количество рекомендаций, которые учитываются при подсчёте метрики.

    Returns:
        Значение метрики AP@K.
    """
    
    # Отрезаем список рекомендаций до k элементов
    recommendations = recommendations[:k]
    
    # Создаем список бинарных меток (1 - книга в рекомендациях есть в списке положительно оцененных, 0 - нет)
    y_true = [1 if book_ind in true_positives else 0 for book_ind in recommendations]
    
    # Используем sklearn.metrics.average_precision_score для подсчета AP@K
    ap = average_precision_score(y_true, [1]*len(y_true)) 
    
    return ap

### Случайный бейзлайн

In [21]:
random_users_inds = np.random.choice(len(train_user_index), size=500, replace=False)
random_baseline_map10 = []

In [22]:
# Получаем список книг, которые пользователь уже прочитал
already_read = []  # Создаем словарь для хранения уже прочитанных книг
  
for rand_user_ind in random_users_inds:
    user_id = train_user_index[rand_user_ind]  # Получаем user_id
    already_read = train_book_index[np.where(train_matrix[rand_user_ind] >= 0)[0]].tolist()  # Получаем все прочитанные книги 
    # Генерируем случайный список из 20 книг
    random_recommendations = np.random.choice(len(train_book_index), size=20, replace=False)
    # Удаляем книги, которые пользователь уже прочитал
    unique_recommendations = list(set(random_recommendations) - set(already_read))
    unique_recommendations = train_book_index[unique_recommendations].tolist()
 
    # Получаем список книг, которые пользователь положительно оценил
    true_positives = test.loc[(test.user_id == user_id) & (test.rating == 1), 'book_id'].tolist()
     
    if unique_recommendations and true_positives: # Проверка на наличие положительных оценок
        # Вычисляем AP@10
        ap10 = ap_at_k(unique_recommendations, true_positives, k=10)
        random_baseline_map10.append(ap10)
    else:
        if not true_positives:
            print(f"Пользователь {user_id} не имеет положительных оценок в тестовой выборке!")
        else:
            print(f"Пользователь {user_id} уже прочитал все книги или не имеет положительных оценок в тестовой выборке!")


Пользователь 13756 не имеет положительных оценок в тестовой выборке!
Пользователь 5422 уже прочитал все книги или не имеет положительных оценок в тестовой выборке!
Пользователь 13437 не имеет положительных оценок в тестовой выборке!
Пользователь 41734 не имеет положительных оценок в тестовой выборке!
Пользователь 25215 не имеет положительных оценок в тестовой выборке!
Пользователь 31746 уже прочитал все книги или не имеет положительных оценок в тестовой выборке!
Пользователь 40075 уже прочитал все книги или не имеет положительных оценок в тестовой выборке!
Пользователь 21841 уже прочитал все книги или не имеет положительных оценок в тестовой выборке!
Пользователь 36964 уже прочитал все книги или не имеет положительных оценок в тестовой выборке!
Пользователь 30558 уже прочитал все книги или не имеет положительных оценок в тестовой выборке!
Пользователь 14334 не имеет положительных оценок в тестовой выборке!
Пользователь 8351 уже прочитал все книги или не имеет положительных оценок в тес

In [23]:
# Среднее значение AP@10
mean_ap_at_10 = np.mean(random_baseline_map10)
print(f"Среднее значение AP@10: {mean_ap_at_10}")

Среднее значение AP@10: 0.0


### Бейзлайн из самых популярных книг

In [19]:
popularity_counts = train.groupby('book_id').size()
top_popular_books = popularity_counts.nlargest(20).index.tolist()

popular_baseline_map10 = []
for rand_user_ind in random_users_inds:
    user_id = train_user_index[rand_user_ind]
    true_positives = test.loc[(test.user_id == user_id) & (test.rating == 1), 'book_id'].tolist()

    if true_positives:
        ap10 = ap_at_k(top_popular_books, true_positives, k=10)
        popular_baseline_map10.append(ap10)
    else:
        print(f"Пользователь {user_id} не имеет положительных оценок в тестовой выборке!")



Пользователь 16628 не имеет положительных оценок в тестовой выборке!
Пользователь 30302 не имеет положительных оценок в тестовой выборке!
Пользователь 4259 не имеет положительных оценок в тестовой выборке!
Пользователь 25082 не имеет положительных оценок в тестовой выборке!
Пользователь 38842 не имеет положительных оценок в тестовой выборке!
Пользователь 3168 не имеет положительных оценок в тестовой выборке!
Пользователь 16132 не имеет положительных оценок в тестовой выборке!
Пользователь 27192 не имеет положительных оценок в тестовой выборке!
Пользователь 40981 не имеет положительных оценок в тестовой выборке!
Пользователь 13310 не имеет положительных оценок в тестовой выборке!
Пользователь 9840 не имеет положительных оценок в тестовой выборке!
Пользователь 324 не имеет положительных оценок в тестовой выборке!
Пользователь 220 не имеет положительных оценок в тестовой выборке!
Пользователь 10376 не имеет положительных оценок в тестовой выборке!
Пользователь 46219 не имеет положительных

In [20]:
mean_popular_ap_at_10 = np.mean(popular_baseline_map10)
print(f"Среднее значение AP@10 для бейзлайна из популярных книг: {mean_popular_ap_at_10}")

Среднее значение AP@10 для бейзлайна из популярных книг: 0.012653061224489797


### Задание 3. Применение метода матричной факторизации и улучшение параметров алгоритма факторизации

In [41]:
user_id = train_user_index[user_index]

In [42]:
# Создание модели ALS
model = implicit.als.AlternatingLeastSquares(factors=64, iterations=30)

model.fit(sparse.csr_matrix(train_matrix), show_progress=True)

# Получаем рекомендации для пользователя
recommendations = model.recommend(user_index, sparse.csr_matrix(train_matrix)[user_index], N=20)

# Преобразуем индексы рекомендаций в book_id
recommended_book_ids = train_book_index[recommendations[0]]

# Выводим рекомендации
print(f"Рекомендации для пользователя {user_id}:")
for book_id in recommended_book_ids:
    print(f"- {book_id}")

  0%|          | 0/30 [00:00<?, ?it/s]

Рекомендации для пользователя 4:
- 133
- 50
- 59
- 117
- 80
- 76
- 15
- 171
- 114
- 241
- 70
- 278
- 90
- 230
- 48
- 740
- 77
- 100
- 26
- 150


In [45]:
from tqdm import tqdm 

In [46]:
# Расчет mAP@10
als_map10 = []
for rand_user_ind in tqdm(random_users_inds):  # Используем tqdm для прогрессбара
    user_id = train_user_index[rand_user_ind]
    true_positives = test[test.user_id == user_id][test.rating == 1].book_id.tolist()

    if true_positives:
        recommendations = model.recommend(rand_user_ind, sparse.csr_matrix(train_matrix)[rand_user_ind], N=20)
        recommended_book_ids = train_book_index[recommendations[0]]
        ap10 = ap_at_k(recommended_book_ids, true_positives, k=10)
        als_map10.append(ap10)

100%|████████████████████████████████████████| 500/500 [10:47<00:00,  1.30s/it]


In [47]:
mean_als_ap_at_10 = np.mean(als_map10)
print(f"Среднее значение AP@10 для ALS: {mean_als_ap_at_10}")

# Сравнение с бейзлайнами
if mean_als_ap_at_10 > mean_popular_ap_at_10 and mean_als_ap_at_10 > mean_ap_at_10:
    print("ALS превзошел оба бейзлайна!")
elif mean_als_ap_at_10 > mean_popular_ap_at_10:
    print("ALS превзошел бейзлайн из популярных книг!")
elif mean_als_ap_at_10 > mean_ap_at_10:
    print("ALS превзошел случайный бейзлайн!")
else:
    print("ALS не превзошел ни один из бейзлайнов.")

Среднее значение AP@10 для ALS: 0.022540983606557378
ALS превзошел оба бейзлайна!


In [49]:
# Разные варианты параметров ALS
for factors in [200, 500]:
    for iterations in [20, 50]:
        print(f"Factors: {factors}, Iterations: {iterations}")
        # Обучаем ALS
        als = implicit.als.AlternatingLeastSquares(factors=factors, regularization=0.01, iterations=iterations)
        als.fit(sparse.csr_matrix(train_matrix), show_progress=True)

        # Расчет mAP@10
        als_map10 = []
        for rand_user_ind in tqdm(random_users_inds):  # Используем tqdm для прогрессбара
            user_id = train_user_index[rand_user_ind]
            true_positives = test[test.user_id == user_id][test.rating == 1].book_id.tolist()
        
            if true_positives:
                recommendations = model.recommend(rand_user_ind, sparse.csr_matrix(train_matrix)[rand_user_ind], N=20)
                recommended_book_ids = train_book_index[recommendations[0]]
                ap10 = ap_at_k(recommended_book_ids, true_positives, k=10)
                als_map10.append(ap10)

        mean_als_ap_at_10 = np.mean(als_map10)
        print(f"Среднее значение AP@10 для ALS: {mean_als_ap_at_10}")

        # Сравнение с бейзлайнами
        if mean_als_ap_at_10 > mean_popular_ap_at_10 and mean_als_ap_at_10 > mean_ap_at_10:
            print("ALS превзошел оба бейзлайна!")
        elif mean_als_ap_at_10 > mean_popular_ap_at_10:
            print("ALS превзошел бейзлайн из популярных книг!")
        elif mean_als_ap_at_10 > mean_ap_at_10:
            print("ALS превзошел случайный бейзлайн!")
        else:
            print("ALS не превзошел ни один из бейзлайнов.")
        print("-" * 20)  # Разделитель между вариантами параметров

Factors: 200, Iterations: 20


  0%|          | 0/20 [00:00<?, ?it/s]

100%|████████████████████████████████████████| 500/500 [10:47<00:00,  1.30s/it]


Среднее значение AP@10 для ALS: 0.022540983606557378
ALS превзошел оба бейзлайна!
--------------------
Factors: 200, Iterations: 50


  0%|          | 0/50 [00:00<?, ?it/s]

...

100%|████████████████████████████████████████| 500/500 [10:46<00:00,  1.29s/it]


Среднее значение AP@10 для ALS: 0.022540983606557378
ALS превзошел оба бейзлайна!
--------------------
Factors: 500, Iterations: 50


  0%|          | 0/50 [00:00<?, ?it/s]

100%|████████████████████████████████████████| 500/500 [10:47<00:00,  1.30s/it]

Среднее значение AP@10 для ALS: 0.022540983606557378
ALS превзошел оба бейзлайна!
--------------------
